# 🎯 Ridge & Lasso (Regularization)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, OLS overfits. Ridge and Lasso add a penalty that shrinks coefficients toward zero — trading a small increase in bias for a big reduction in variance.

### What you'll learn
- Why OLS fails with many or correlated predictors
- Ridge regression (L2 penalty) — shrinks all coefficients
- Lasso regression (L1 penalty) — shrinks *and* zeros out coefficients
- How to choose λ using cross-validation
- When to use Ridge vs Lasso

### Dataset: Hitters (baseball salary prediction, 20 predictors)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':11
})
print("✓ Setup complete")
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Hitters

In [ ]:
import pandas as pd, numpy as np
hitters = pd.read_csv(f'{DATA_DIR}/Hitters.csv').dropna()
hitters = pd.get_dummies(hitters, drop_first=True, dtype=float)
X = hitters.drop('Salary', axis=1)
y = np.log(hitters['Salary'])
print(f"Hitters: {X.shape}  |  Predicting: log(Salary)")

## 🎯 Part 1 — The Problem: OLS With Many Predictors

With 20 predictors, OLS fits *perfectly* on training data but generalizes poorly. When predictors are correlated, OLS coefficients become unstable — small changes in data cause large swings in estimates.

**Regularization** adds a penalty term to the OLS objective:

**Ridge (L2):** minimize RSS + λ × Σβⱼ²  
**Lasso (L1):** minimize RSS + λ × Σ|βⱼ|

λ (lambda) controls penalty strength:
- λ=0 → same as OLS  
- λ→∞ → all coefficients → 0

In [ ]:
# Show coefficient paths as lambda varies
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

alphas = np.logspace(-3, 5, 200)
ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr_s, y_tr).coef_)
    try:
        lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_tr_s, y_tr).coef_)
    except:
        lasso_coefs.append(np.zeros(X_tr_s.shape[1]))

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(ridge_coefs.shape[1]):
    axes[0].plot(np.log10(alphas), ridge_coefs[:,i], lw=0.8, alpha=0.7)
axes[0].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[0].set_xlabel('log₁₀(λ)')
axes[0].set_ylabel('Standardized Coefficient')
axes[0].set_title('Ridge — coefficients shrink smoothly\n(none reach exactly zero)')

for i in range(lasso_coefs.shape[1]):
    axes[1].plot(np.log10(alphas), lasso_coefs[:,i], lw=0.8, alpha=0.7)
axes[1].axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
axes[1].set_xlabel('log₁₀(λ)')
axes[1].set_ylabel('Standardized Coefficient')
axes[1].set_title('Lasso — coefficients reach exactly zero\n(automatic feature selection!)')

plt.suptitle('Coefficient Paths: Ridge vs Lasso', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Key difference: Lasso produces sparse solutions (some β=0 exactly).")
print("   Ridge keeps all predictors but shrinks them.")

In [ ]:
# Choose optimal lambda via cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-3, 5, 100), cv=10)
lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 100), cv=10, max_iter=10000)

ridge_cv.fit(X_tr_s, y_tr)
lasso_cv.fit(X_tr_s, y_tr)

print(f"Ridge optimal λ: {ridge_cv.alpha_:.4f}")
print(f"Lasso optimal λ: {lasso_cv.alpha_:.4f}")
print(f"Lasso zeroed out: {np.sum(lasso_cv.coef_ == 0)}/{X.shape[1]} predictors")

# Compare test MSE
ols   = LinearRegression().fit(X_tr_s, y_tr)
models_res = {
    'OLS':   mean_squared_error(y_te, ols.predict(X_te_s)),
    'Ridge': mean_squared_error(y_te, ridge_cv.predict(X_te_s)),
    'Lasso': mean_squared_error(y_te, lasso_cv.predict(X_te_s)),
}

print("\n=== Test MSE Comparison ===")
for name, mse in models_res.items():
    print(f"  {name:<8} MSE = {mse:.4f}  {'← best' if mse == min(models_res.values()) else ''}")

In [ ]:
# Visualize: which predictors Lasso keeps vs zeros
coef_df = pd.DataFrame({
    'Predictor': X.columns,
    'OLS': ols.coef_,
    'Ridge': ridge_cv.coef_,
    'Lasso': lasso_cv.coef_
}).sort_values('Lasso', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(coef_df))
w = 0.28
ax.bar(x_pos - w, coef_df['OLS'],   w, label='OLS',   color='#888',    alpha=0.7)
ax.bar(x_pos,     coef_df['Ridge'], w, label='Ridge',  color='#1e5fa8', alpha=0.8)
ax.bar(x_pos + w, coef_df['Lasso'], w, label='Lasso',  color='#e85d20', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(coef_df['Predictor'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Standardized Coefficient')
ax.set_title('OLS vs Ridge vs Lasso Coefficients')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Orange bars at zero = Lasso has eliminated that predictor entirely.")

In [ ]:
answers = {
    "q1": "",  # What penalty does Ridge add to the OLS objective?
    "q2": "",  # Why does Lasso produce exactly-zero coefficients but Ridge does not?
    "q3": "",  # How do you choose the optimal λ in practice?
    "q4": "",  # When would you prefer Ridge over Lasso?
    "q5": "",  # What happens to all coefficients as λ → ∞?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")